In [ ]:
# ==========================================
# BLOCK 1: ENVIRONMENT SETUP & PERSISTENCE
# ==========================================
# @title ⚙️ 1. Setup Environment & Storage

# @markdown ### 📂 Workspace Configuration
DRIVE_WORKSPACE_NAME = "AutoScribe_Workspace" # @param {type:"string"}
SHARED_DRIVE_NAME = "" # @param {type:"string"}

import os
import sys
import shutil
from datetime import datetime
from google.colab import drive
from IPython.display import clear_output, display
from IPython.utils import capture
import ipywidgets as widgets

# --- LOCAL I/O SETUP (Maximum throughput) ---
LOCAL_WORKSPACE = "/content/AutoScribe_Local"
TEMP_DIR = f"{LOCAL_WORKSPACE}/temp_media"
LOCAL_OUTPUT = f"{LOCAL_WORKSPACE}/outputs"
LOCAL_LOG = f"{LOCAL_WORKSPACE}/autoscribe_debug.log"

for directory in [TEMP_DIR, LOCAL_OUTPUT]:
    os.makedirs(directory, exist_ok=True)

def log_event(level, message, print_to_console=True):
    if print_to_console: 
        print(message, flush=True)
    try:
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        with open(LOCAL_LOG, 'a', encoding='utf-8') as f:
            f.write(f"[{timestamp}] [{level}] {message}\n")
    except Exception: pass

def inf(msg, style, wdth):
    display(widgets.Button(description=msg, disabled=True, button_style=style, layout=widgets.Layout(min_width=wdth)))

print("Mounting Google Drive...", flush=True)
drive.mount('/content/drive', force_remount=True)

if SHARED_DRIVE_NAME != "" and os.path.exists(f"/content/drive/Shareddrives/{SHARED_DRIVE_NAME}"):
    root_path = f"/content/drive/Shareddrives/{SHARED_DRIVE_NAME}"
else:
    root_path = "/content/drive/MyDrive"

workspace_name = DRIVE_WORKSPACE_NAME.strip() or "AutoScribe_Workspace"
DRIVE_BASE = f"{root_path}/{workspace_name}"
DRIVE_OUTPUT_DIR = os.path.join(DRIVE_BASE, "outputs")

# We only ensure the output directory exists on Drive. No caching on Drive!
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

log_event("INFO", "=== NEW AUTOSCRIBE SESSION STARTED ===", print_to_console=False)

# Local cache for HuggingFace models
os.environ["HF_HOME"] = "/content/local_hf_cache"

with capture.capture_output() as cap:
    log_event("INFO", "Installing dependencies to LOCAL runtime...", print_to_console=False)
    # Installing to local Colab environment, NOT Drive.
    !pip install -q yt-dlp faster-whisper ffmpeg-python transformers accelerate torch torchvision
    !pip install -q --upgrade --no-deps yt-dlp

BLOCK_1_COMPLETED = True
clear_output()
inf('\u2714 Workspace Ready (Local NVMe Architecture Active)', 'success', '350px')

In [ ]:
# ==========================================
# BLOCK 2: INGESTION & ROUTING
# ==========================================
# @title 📥 2. Source Configuration & Routing

# @markdown ### ⚙️ 1. Select Source
SOURCE_TYPE = "URL" # @param ["URL", "Google_Drive"]

# @markdown ---
# @markdown ### 🔗 Option A: URL
YOUTUBE_URL = "https://youtube.com/playlist?list=..." # @param {type:"string"}

# @markdown ### 📂 Option B: Google Drive
DRIVE_FILE_PATH = "/content/drive/MyDrive/your_video.mp4" # @param {type:"string"}
# @markdown ---

# @markdown ### 🧠 2. AI Processing Settings
WHISPER_MODE = "Auto" # @param ["On", "Off", "Auto"]
VISION_FALLBACK = True # @param {type:"boolean"}

if 'BLOCK_1_COMPLETED' not in locals():
    raise RuntimeError("🚨 ERROR: Run Block 1 first.")

import yt_dlp
import subprocess
import shutil
import os

routing_queue = []

def analyze_and_route(file_path, title, video_id):
    log_event("INFO", f"Analyzing audio stream for: {title}...")
    command = ["ffprobe", "-v", "error", "-select_streams", "a", "-show_entries", "stream=codec_type", "-of", "default=nw=1:nk=1", file_path]
    requires_vision = False
    
    try:
        result = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        has_audio = "audio" in result.stdout.strip().lower()
        
        if WHISPER_MODE == "Auto":
            if not has_audio and VISION_FALLBACK:
                requires_vision = True
                log_event("INFO", "[ROUTING]: No audio stream detected. Tagged for Vision analysis.")
            else:
                log_event("INFO", "[ROUTING]: Audio stream intact. Tagged for Whisper transcription.")
        elif WHISPER_MODE == "Off" and VISION_FALLBACK:
            requires_vision = True
            log_event("INFO", "[ROUTING]: Whisper mode forced OFF. Tagged for Vision analysis.")
            
    except Exception as e:
        log_event("ERROR", f"Audio analysis failed for {title}: {e}. Fallback to Vision analysis.")
        requires_vision = True

    routing_queue.append({'id': video_id, 'title': title, 'path': file_path, 'use_vision': requires_vision})

log_event("INFO", f"Processing source type: {SOURCE_TYPE}")

if SOURCE_TYPE == "Google_Drive":
    if os.path.exists(DRIVE_FILE_PATH):
        filename = os.path.basename(DRIVE_FILE_PATH)
        dest_path = os.path.join(TEMP_DIR, filename)
        log_event("INFO", f"Copying {filename} from Google Drive to local NVMe storage...")
        shutil.copy2(DRIVE_FILE_PATH, dest_path)
        analyze_and_route(dest_path, filename, filename.split('.')[0])
    else:
        log_event("ERROR", f"File not found on Google Drive. Please verify the path: {DRIVE_FILE_PATH}")

elif SOURCE_TYPE == "URL":
    ydl_opts_meta = {'extract_flat': 'in_playlist', 'quiet': True}
    with yt_dlp.YoutubeDL(ydl_opts_meta) as ydl:
        try:
            info = ydl.extract_info(YOUTUBE_URL, download=False)
            entries = info['entries'] if 'entries' in info else [info]
            for entry in entries:
                v_id = entry['id']
                title = entry.get('title', v_id)
                url = entry.get('url', YOUTUBE_URL)
                log_event("INFO", f"Downloading: {title}")
                
                ydl_opts_dl = {
                    'format': 'bestaudio/best',
                    'outtmpl': f'{TEMP_DIR}/{v_id}.%(ext)s',
                    'quiet': True
                }
                with yt_dlp.YoutubeDL(ydl_opts_dl) as ydl_dl:
                    dl_info = ydl_dl.extract_info(url, download=True)
                    local_file = ydl_dl.prepare_filename(dl_info)
                    analyze_and_route(local_file, title, v_id)
        except Exception as e:
            log_event("ERROR", f"URL parsing failed: {e}")

BLOCK_2_COMPLETED = True
log_event("INFO", f"✅ Block 2 Complete. {len(routing_queue)} item(s) in queue.")

In [ ]:
# ==========================================
# BLOCK 3: AI PROCESSING & LOCAL EXPORT
# ==========================================
# @title 🧠 3. Execute AI Processing

# @markdown *This block initializes the AI models, processes the media files locally, and compiles the Markdown outputs.*
# @markdown *Note: The first run may take a few minutes as the AI models are downloaded to the local cache.*

import sys
print("🚀 INITIALIZING BLOCK 3 - Loading libraries locally...", flush=True)

import gc
import torch
import subprocess
import os
import re
from datetime import datetime
from PIL import Image

if 'BLOCK_2_COMPLETED' not in locals():
    raise RuntimeError("🚨 ERROR: Run Block 1 and 2 before executing AI processing.")
if not routing_queue:
    raise ValueError("🚨 ERROR: Routing queue is empty. No source files found.")

try:
    from faster_whisper import WhisperModel
    print("✅ faster_whisper imported successfully.", flush=True)
except ImportError:
    raise RuntimeError("🚨 ERROR: faster_whisper library is missing.")

print("⚠️ RETRIEVING HF_TOKEN... (If execution hangs here, ensure 'Notebook access' is enabled in the Secrets tab!)", flush=True)
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
    print("✅ Hugging Face token authenticated.", flush=True)
except Exception as e:
    raise PermissionError(f"CRITICAL ERROR: 'HF_TOKEN' retrieval failed - {e}")

LOCAL_MODEL_CACHE = "/content/local_hf_cache"
os.makedirs(LOCAL_MODEL_CACHE, exist_ok=True)
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

run_timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
LOCAL_EXPORT_DIR = os.path.join(LOCAL_OUTPUT, f"AutoScribe_Run_{run_timestamp}")
os.makedirs(LOCAL_EXPORT_DIR, exist_ok=True)

audio_tasks = [t for t in routing_queue if not t['use_vision']]
vision_tasks = [t for t in routing_queue if t['use_vision']]

def get_safe_filename(title):
    safe_title = re.sub(r'[^\w\s-]', '', title).strip().replace(' ', '_')
    return safe_title[:50]

if audio_tasks:
    log_event("INFO", "\n--- Loading Whisper Model ---")
    try:
        whisper_model = WhisperModel("large-v3", device="cuda", compute_type="float16", download_root=LOCAL_MODEL_CACHE)
        for task in audio_tasks:
            log_event("INFO", f"🎙️ Transcribing: {task['title']} (Writing to local NVMe)")
            safe_name = get_safe_filename(task['title'])
            export_path = os.path.join(LOCAL_EXPORT_DIR, f"{safe_name}_audio.md")
            
            try:
                segments, info = whisper_model.transcribe(task['path'], beam_size=5, language="sv", condition_on_previous_text=False)
                with open(export_path, "w", encoding="utf-8") as f:
                    f.write(f"# Source: {task['title']}\n**Analysis Type:** Audio Transcription\n\n")
                    for s in segments:
                        f.write(f"[{divmod(int(s.start), 60)[0]:02d}:{divmod(int(s.start), 60)[1]:02d}] {s.text.strip()}\n")
            except Exception as e: log_event("ERROR", f"Whisper transcription failed: {e}")
    except Exception as e: log_event("ERROR", f"Model initialization failed: {e}")
    finally:
        if 'whisper_model' in locals(): del whisper_model
        gc.collect()
        torch.cuda.empty_cache()

if vision_tasks:
    log_event("INFO", "\n--- Loading Moondream2 Vision Model ---")
    try:
        from transformers import AutoModelForCausalLM, AutoTokenizer
        model_id = "vikhyatk/moondream2"
        tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True, cache_dir=LOCAL_MODEL_CACHE)
        moondream = AutoModelForCausalLM.from_pretrained(model_id, trust_remote_code=True, cache_dir=LOCAL_MODEL_CACHE).to("cuda")
        moondream.eval()

        for task in vision_tasks:
            log_event("INFO", f"👁️ Processing Visuals: {task['title']}")
            safe_name = get_safe_filename(task['title'])
            export_path = os.path.join(LOCAL_EXPORT_DIR, f"{safe_name}_vision.md")
            
            frame_dir = os.path.join(TEMP_DIR, f"frames_{task['id']}")
            os.makedirs(frame_dir, exist_ok=True)
            subprocess.run(["ffmpeg", "-i", task['path'], "-vf", "fps=1/30", f"{frame_dir}/frame_%04d.jpg", "-hide_banner", "-loglevel", "error"])
            
            frames = sorted(os.listdir(frame_dir))
            
            with open(export_path, "w", encoding="utf-8") as f:
                f.write(f"# Source: {task['title']}\n**Analysis Type:** Visual Frame Analysis\n\n")
                for i, frame_file in enumerate(frames):
                    try:
                        img_path = os.path.join(frame_dir, frame_file)
                        enc_image = moondream.encode_image(Image.open(img_path))
                        answer = moondream.answer_question(enc_image, "Describe the key action or text in this frame concisely.", tokenizer)
                        f.write(f"[{divmod(i * 30, 60)[0]:02d}:{divmod(i * 30, 60)[1]:02d}] {answer}\n")
                    except Exception as e: log_event("WARNING", f"Skipping frame due to error: {e}")
                    
    except Exception as e: log_event("ERROR", f"Vision Model failed: {e}")
    finally:
        if 'moondream' in locals(): del moondream
        gc.collect()
        torch.cuda.empty_cache()

log_event("INFO", "\n🚀 Pushing compiled data to Google Drive...")
DRIVE_FINAL_DIR = os.path.join(DRIVE_OUTPUT_DIR, f"AutoScribe_Run_{run_timestamp}")
shutil.copytree(LOCAL_EXPORT_DIR, DRIVE_FINAL_DIR)

BLOCK_3_COMPLETED = True
log_event("INFO", f"✅ Block 3 Complete. Output securely saved to Drive: {DRIVE_FINAL_DIR}")

In [ ]:
# ==========================================
# BLOCK 4: CLEANUP & SHUTDOWN
# ==========================================
# @title 📝 4. Finalize & Disconnect

# @markdown *This block performs essential housekeeping. It transfers the debug logs to your Drive, securely deletes temporary files, and disconnects the runtime to preserve your Compute Units.*

from google.colab import runtime

if 'BLOCK_3_COMPLETED' not in locals():
    raise RuntimeError("🚨 ERROR: Cannot finalize session. Block 3 must run successfully first.")

log_event("INFO", "--- Finalizing Processing Run ---")
try:
    if os.path.exists(LOCAL_LOG):
        shutil.copy2(LOCAL_LOG, os.path.join(DRIVE_OUTPUT_DIR, "autoscribe_debug.log"))
    
    if os.path.exists(LOCAL_WORKSPACE):
        shutil.rmtree(LOCAL_WORKSPACE)
        log_event("INFO", "🗑️ Local workspace and temporary media caches cleared.")
except Exception as e:
    print(f"Cleanup warning: {e}")

print("🎉 All processing complete. Files and logs are safely stored on Google Drive.", flush=True)
print("Disconnecting runtime to save Compute Units.", flush=True)

runtime.unassign()